In [0]:
def handle_error(e, dbutils=None, max_len=1000):
    import traceback
    from datetime import datetime
    error_type = type(e).__name__
    error_summary = str(e)
    error_trace = traceback.format_exc()

    error_msg_full = f"""
        ERROR_TYPE: {error_type}
        ERROR_SUMMARY: {error_summary}
        TIMESTAMP: {datetime.now()}
        TRACEBACK:
        {error_trace}
        """

    # truncar si es muy largo
    if len(error_msg_full) > max_len:
        error_msg = error_msg_full[:max_len] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full
    
    dbutils.jobs.taskValues.set(key="error", value=error_msg)
    return error_msg

In [0]:
def no_registros(df, columna, lista, tabla, fecha):
    from pyspark.sql.functions import col
    registros = df.filter(col(columna).isNull()).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_registros",
        "cumple":registros > 0,
        "detalle":f"No hay registros nuevos para la fecha: {fecha}"
    })

In [0]:
def nulos(df, columnas, lista, tabla):
    from pyspark.sql.functions import col
    from functools import reduce
    
    if isinstance(columnas, str):
        columnas = [columnas]
    
    condicion = reduce(lambda acc, c: acc | col(c).isNull(), columnas[1:], col(columnas[0]).isNull())
    
    null = df.filter(condicion).count()
    
    lista.append({
        "tabla": tabla,
        "columna": ", ".join(columnas),
        "regla": "no_null",
        "cumple": null == 0,
        "detalle": f"Registros nulos: {null}"
    })

In [0]:
def duplicados(df, columnas, lista, tabla):
    from pyspark.sql.functions import col
    
    if isinstance(columnas, str):
        columnas = [columnas]
    
    group_cols = [col(c) for c in columnas]
    
    duplicates = (
        df.groupBy(*group_cols)
          .count()
          .filter(col("count") > 1)
          .count()
    )
    
    lista.append({
        "tabla": tabla,
        "columna": ", ".join(columnas),
        "regla": "no_duplicate",
        "cumple": duplicates == 0,
        "detalle": f"Registros duplicados: {duplicates}"
    })

In [0]:
def nulosx(df, columna, lista, tabla):
    from pyspark.sql.functions import col
    null = df.filter(col(columna).isNull()).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_null",
        "cumple":null == 0,
        "detalle":f"Registros nulos:{null}"
    })

In [0]:
def duplicadosx(df, columna, lista, tabla):
    from pyspark.sql.functions import col
    duplicates = df.groupBy(col(columna)).count().filter(col("count") > 1).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_duplicate",
        "cumple":duplicates == 0,
        "detalle":f"Registros duplicados:{duplicates}"
    })